## House Price Prediction using a Feedforward Neural Network

This notebook builds a small neural network using Keras to predict house prices
from two features: number of bedrooms and living area (sqft).

**Model Architecture:**
- Input layer: 2 features (bedrooms, sqft_living)
- Hidden layer: 2 neurons, Sigmoid activation
- Output layer: 1 neuron, Linear activation (regression output)
- Total trainable parameters: **just 9**

**Training:** SGD optimizer, Mean Squared Error loss, 10 epochs, batch size 32

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## Part 1: Import the Housing data and do feature transformations

In [2]:
df= pd.read_csv('house_price_full.csv')
df.head()

,bedrooms,sqft_living,price
0,3,1340,313000
1,5,3650,2384000
2,3,1930,342000
3,3,2000,420000
4,4,1940,550000


In [3]:
X = df.copy()
# Remove target
Y = X.pop('price')

# perform a scaler transform of the input data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# perform log transformation of target variable (For Sandeep: Is this needed?)
Y = np.log(Y)

In [4]:
df_scaled = pd.DataFrame(X)
df_scaled

,0,1
0,-0.433198,-0.753258
1,1.675735,1.457330
2,-0.433198,-0.188649
3,-0.433198,-0.121661
4,0.621269,-0.179079
...,...,...
494,0.621269,0.873582
495,1.675735,2.299459
496,-0.433198,-0.724549
497,-0.433198,-0.179079


In [5]:
Y

,price
0,12.653958
1,14.684290
2,12.742566
3,12.948010
4,13.217674
...,...
494,13.380102
495,13.764217
496,12.128111
497,12.721886


## Part 2: Create Model Using `keras`

![](multiple_neurons.png)

In [6]:
from tensorflow import keras

In [8]:
#Creating model with 2 layers n having act fun as sigmoid
model = keras.Sequential(
    [
        keras.layers.Dense(
            2, activation="sigmoid", input_shape=(X.shape[-1],)
        ),
        keras.layers.Dense(1, activation="linear")
    ]
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

```python
def random_init_params():
    w1 = tf.Variable(tf.random.uniform((2, 2)))
    b1 = tf.Variable(tf.random.uniform((1, 2)))
    w2 = tf.Variable(tf.random.uniform((2, 1)))
    b2 = tf.Variable(tf.random.uniform((1, 1)))
    return w1,b1,w2,b2


def forward_prop(x, w1, b1, w2, b2):
    z1 = tf.matmul(x,w1) + b1
    h1 = tf.math.sigmoid(z1)
    z2 = tf.matmul(h1,w2) + b2
    h2 = z2
    return h2
```

# Part 3: Training

In [10]:
model.compile(
    optimizer=keras.optimizers.SGD(), loss="mean_squared_error"
)

In [13]:

def train(x, y, w1, b1, w2, b2):
    y_true = y
    with tf.GradientTape() as g:
        y_pred = forward_prop(x, w1, b1, w2, b2)

        # loss
        loss = 0.5*(y_true - y_pred)** 2

    #Gradient calculation
    print("**************************************************")
    print("GRADIENTS")
    print("**************************************************")
    gw1, gb1, gw2, gb2 = g.gradient(loss, [w1, b1, w2, b2])
    print(" the gradient for 1st layer weights are:\n",gw1.numpy())
    print("--------------------------------------------------")
    print(" the gradient for 2nd layer weights are:\n",gw2.numpy())
    print("--------------------------------------------------")
    print(" the gradient for 1st layer bias are:\n",gb1.numpy())
    print("--------------------------------------------------")
    print(" the gradient for 2nd layer bias are:\n",gb2.numpy())
    print("--------------------------------------------------")

    # Gradient descent:
    lr=0.2
    w1.assign_sub(lr*gw1)
    b1.assign_sub(lr*gb1)
    w2.assign_sub(lr*gw2)
    b2.assign_sub(lr*gb2)
    print("**************************************************")
    print("NEW UPDATES")
    print("**************************************************")
    print(" the updated 1st layer weights are:\n",w1.numpy())
    print("--------------------------------------------------")
    print(" the updated 2nd layer weights are:\n",w2.numpy())
    print("--------------------------------------------------")
    print(" the updated 1st layer bias are:\n",b1.numpy())
    print("--------------------------------------------------")
    print(" the updated 2nd layer bias are:\n",b2.numpy())


    return w1, b1, w2, b2,loss



In [14]:
model.fit(X,Y.values,epochs=10,batch_size=32)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2182 
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2073 
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1978 
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1902 
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.1845
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1795  
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1761 
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1722 
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1692 
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1666 


In [15]:
model.predict(X)[:,0]

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


array([12.893007 , 13.638782 , 13.075309 , 13.090449 , 13.141019 ,
       12.558638 , 12.668678 , 13.392218 , 13.157402 , 12.929815 ,
       13.019167 , 13.4347925, 13.145593 , 12.783295 , 13.442567 ,
       12.834097 , 12.976301 , 13.46529  , 12.90481  , 12.825158 ,
       13.133089 , 12.889016 , 13.030373 , 13.396698 , 12.974459 ,
       13.108083 , 13.149361 , 13.244199 , 13.409744 , 13.172729 ,
       12.952401 , 13.423616 , 13.086448 , 13.415735 , 13.515316 ,
       13.184418 , 12.533308 , 13.056426 , 13.081953 , 13.152935 ,
       12.979555 , 12.9067   , 12.5493765, 13.035805 , 12.642509 ,
       12.816088 , 13.084116 , 13.075309 , 12.864347 , 13.489941 ,
       13.392218 , 13.186717 , 13.179657 , 13.043745 , 13.125795 ,
       13.1322365, 13.240599 , 13.640645 , 12.88498  , 12.705819 ,
       13.489941 , 13.17799  , 13.396698 , 13.167978 , 13.4347925,
       12.564655 , 13.001499 , 12.743811 , 12.702579 , 13.12694  ,
       13.347605 , 13.185317 , 13.271986 , 13.068428 , 12.8890

## Key Findings

**Training:** Loss dropped from **0.2182 → 0.1666** over 10 epochs show steady decline,
no overfitting, confirming the network is learning a real price pattern.

**Predictions are in log-space** convert back with `e^prediction`:
- 13.0 → ~$442,000 | 13.7 → ~$890,000

**Why log-transform price?** Raw prices ($100K–$3.2M) cause unstable gradiets.
Log compreses them into a tight 12–14 range, making training smooth.

**Limitation:** With only 9 parameters and 2 features, this is a learning exercise
a real model would need more features and deeper layers.